# A vortex dipole collides with a wall

Two counter-rotating vortices approach the top wall of a square box. We compare how free slip and no slip affect their rebound and the energy dissipated near the wall.

## 1. Equations of motion in the square

In the unit square, $\mathbf u=(u,v)$ is velocity, $p$ is pressure per unit density, and $\nu=0.001$ is kinematic viscosity. The unforced incompressible Navier–Stokes equations are
$$
\partial_t\mathbf u+(\mathbf u\cdot\nabla)\mathbf u=-\nabla p+\nu\Delta\mathbf u,\qquad \nabla\cdot\mathbf u=0.
$$
The streamfunction $\psi$ gives velocity and signed vorticity $\omega$:
$$
(u,v)=(\partial_y\psi,-\partial_x\psi),\qquad \omega=\partial_xv-\partial_yu=-\Delta\psi.
$$
Taking the curl gives the equation the solver advances:
$$
\partial_t\omega+u\,\partial_x\omega+v\,\partial_y\omega=\nu\Delta\omega.
$$
The flow transports vorticity, viscosity diffuses it, and solving $-\Delta\psi=\omega$ recovers the velocity field through $\psi$.

## 2. Creating an upward-moving vortex pair

At time zero, two streamfunction bumps set the velocity through $(u_0,v_0)=(\partial_y\psi_0,-\partial_x\psi_0)$. For $z=(x,y)$, their centers are $c_L=(0.44,0.52)$ and $c_R=(0.56,0.52)$:
$$
B(r)=\left[\max\left(1-\frac{r^2}{0.19^2},0\right)\right]^8,\qquad
\psi_0(z)=0.17\big[B(|z-c_L|)-B(|z-c_R|)\big].
$$
Each bump vanishes outside radius $0.19$. Both runs start from the same grid vorticity $\omega_0=-\Delta_h\psi_0$.

The counterclockwise left core drives fluid upward at the right core, and the clockwise right core does the same at the left. Between them $\psi_0$ decreases to the right, so $v=-\partial_x\psi_0>0$. Each vortex carries the other upward without external forcing.

## 3. Wall conditions in the two runs

All four walls are stationary and impermeable, so $\mathbf u\cdot\mathbf n=0$ for the outward normal $\mathbf n$. The streamfunction is therefore constant along the boundary, and we choose $\psi=0$ there. The left, right, and bottom walls have free slip in both runs. Only the top wall at $y=1$ changes.

| Upper wall | Conditions | What the fluid can do |
|---|---|---|
| Free slip | $v=0$, $\partial_y u=0$ | Move along the wall with zero tangential stress. |
| No slip | $v=0$, $u=0$ | Match the wall's zero velocity in both directions. |

Both bumps vanish near every boundary, so the initial velocity satisfies either top-wall condition. The cell below checks that the two runs start with the same velocity and vorticity on the movie grid.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from scipy.integrate import trapezoid

from simulations.run_2d_wall_collision import matched_pair, sampled_runs
from simulations.run_2d_wall_dissipation import compare

free, rigid = matched_pair(193)
np.testing.assert_allclose(free.velocity(), rigid.velocity(), atol=1e-12)
np.testing.assert_allclose(free.vorticity(), rigid.vorticity(), atol=1e-12)
for case in (free, rigid):
    print(
        f"{case.walls['top'].kind:9s}  initial kinetic energy = {case.kinetic_energy():.6f}"
    )

### 3.1 Reading rotation and velocity at the start

The box has side length $L=1$. The coordinates $x/L$ and $y/L$ measure box lengths from the left and bottom walls, so they match $x$ and $y$ numerically here. The gap to the top wall is $1-y/L$ box lengths: at $y/L=0.925$, the gap is $0.075$.

Arrows show velocity. Color shows vorticity $\omega$, in inverse time: red is counterclockwise and blue clockwise. A tiny paddle wheel spins at angular velocity $\omega/2$. On a grid of spacing $h$, the initial color at each interior node comes from its streamfunction and four neighbors:
$$
\omega_{i,j}=\frac{4\psi_{i,j}-\psi_{i+1,j}-\psi_{i-1,j}-\psi_{i,j+1}-\psi_{i,j-1}}{h^2}.
$$
At the midpoint $(0.50,0.52)$, the upward velocities add while the opposite vorticities cancel by symmetry. The upward arrow on the neutral centerline shows motion with zero local spin.

In [ ]:
omega0 = free.vorticity()
u0, v0 = free.velocity()
fig, ax = plt.subplots(figsize=(6.4, 4.4), layout="constrained")
limit = np.abs(omega0).max()
field = ax.pcolormesh(
    free.X, free.Y, omega0, cmap="RdBu_r", vmin=-limit, vmax=limit, shading="auto"
)
# Include the centerline between the cores in the arrow sample.
sample = (slice(None, None, 4), slice(None, None, 4))
ax.quiver(
    free.X[sample],
    free.Y[sample],
    u0[sample],
    v0[sample],
    angles="xy",
    scale_units="xy",
    scale=110,
    pivot="mid",
    color="black",
    width=0.004,
    minlength=0,
)
ax.set(
    xlim=(0.30, 0.70),
    ylim=(0.39, 0.65),
    aspect="equal",
    xlabel="x",
    ylabel="y",
    title="Initial vorticity and velocity near the pair",
)
fig.colorbar(field, ax=ax, label="Signed vorticity $\\omega$")
plt.show()

## 4. Collision at the upper wall

Both panels use the same clock and vorticity scale (clipped at ±75).

**Approach.** The cores rise together, then separate along the wall: red moves left and blue moves right.

**Wall layer visible ($t\approx0.55$).** In the no-slip panel, look for the thin blue band above the red core and the red band above the blue core.

**Rebound ($t\approx0.90$).** Follow the original cores. The free-slip cores have spread farther sideways. The no-slip cores have turned away from the wall.

[Play the collision movie](figures/walls/vortex_wall_collision.mp4)

In [ ]:
from IPython.display import Video

Video(filename="figures/walls/vortex_wall_collision.mp4", embed=True, width=850)

## 5. Wall shear generates opposite-signed vorticity

At $t=0.55$, find the blue band above the red core in the no-slip movie. Both panels below cut through it at $x\approx0.43$. Read the dots upward from $y=0.84$ to the top wall at $y=1$. Each dot is a grid value, and the vertical gray line marks zero.

In [ ]:
for t, free_case, rigid_case in sampled_runs(129):
    if t >= 0.55:
        break

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True, layout="constrained")
for case, label, color in (
    (free_case, "Free slip", "#126987"),
    (rigid_case, "No slip", "#b04729"),
):
    i = np.abs(case.X[:, 0] - 0.43).argmin()
    upper = case.Y[i] >= 0.84
    y = case.Y[i, upper]
    u, _ = case.velocity()
    for ax, field in zip(axes, (u, case.vorticity())):
        ax.plot(field[i, upper], y, ".-", color=color, label=label)
for ax, xlabel in zip(axes, ("Horizontal velocity", "Signed vorticity $\\omega$")):
    ax.axvline(0, color="0.7", linewidth=0.8, zorder=0)
    ax.axhline(1, color="0.3", linewidth=0.8)
    ax.set(xlabel=xlabel, ylim=(0.84, 1.01))
axes[0].set_ylabel("Height y (wall at 1)")
axes[0].legend()
fig.suptitle(f"Above the left core at x = {case.X[i, 0]:.3f}, t = {t:.2f}")
plt.show()

**Left panel — velocity.** Above the red core, the fluid moves left ($u<0$). Follow the no-slip dots toward the wall: $u$ rises sharply to zero, so $\partial_y u>0$. With free slip, the fluid still moves left at the wall, where $\partial_y u=0$.

**Right panel — vorticity.** At those same heights, the negative no-slip values trace the movie’s blue band. Since $v=0$ along the flat wall, $\partial_xv=0$ there and
$$
\omega=\partial_x v-\partial_y u=-\partial_y u.
$$
The positive shear above the red core gives negative wall vorticity. Above the blue core the shear reverses, producing the red band. The free-slip curve reaches zero at the wall.

At the stationary no-slip wall, tangential momentum balance gives $\partial_x p=-\nu\,\partial_y\omega$: the pressure gradient sets the diffusive vorticity flux. [Wall balance, eq. (11)](#ref-kramer) · [Profiles](figures/walls/impact_shear_profiles.png)

## 6. The no-slip cores rebound from the wall

The free-slip cores spread along the wall. The no-slip cores turn back toward the interior. This motion is consistent with the rebound mechanism described by Kramer and colleagues. In that mechanism, part of the wall layer lifts and curls around each core. A clockwise patch on the left flank of the red core induces a downward velocity there, with a mirror effect at the blue core. [Rebound mechanism, sections IV–V](#ref-kramer)

The left panel tracks each core by the strongest interior vorticity extremum of its original sign within $0.04L$ of its previous position. This follows a rotation center rather than a fluid parcel and excludes wall nodes.

The right panel averages the wall distance $d=L-y_c$ over the two cores. Rebound is the largest rise from closest approach through $t=1.20$. We mark visible divergence at the first of three consecutive snapshots with mean cross-case core separation of at least $0.02L$. [Core trajectories](figures/walls/vortex_core_paths.png)

In [ ]:
from simulations.run_2d_wall_collision import closest_approach, core_paths

fig, (paths, gaps) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for n in (129, 193):
    t, xy = core_paths(n)
    distance, closest = closest_approach(xy)
    separation = np.linalg.norm(xy[:, 0] - xy[:, 1], axis=-1).mean(axis=1)
    above = separation >= 0.02
    first = np.flatnonzero(above[:-2] & above[1:-1] & above[2:])[0]
    farthest = closest + distance[closest:, 1].argmax()
    rebound = distance[farthest, 1] - distance[closest, 1]
    print(
        f"{n}²: separation threshold at t={t[first]:.2f}; no-slip gap "
        f"{distance[closest, 1]:.3f} L at t={t[closest]:.2f} to "
        f"{distance[farthest, 1]:.3f} L at t={t[farthest]:.2f}; rebound {rebound:.3f} L"
    )
    for case, (name, color) in enumerate(
        (("Free slip", "#126987"), ("No slip", "#b04729"))
    ):
        gaps.plot(
            t,
            distance[:, case],
            color=color,
            linestyle="-" if n == 193 else ":",
            label=f"{name}, {n}²",
        )
        if n == 193:
            for core in (0, 1):
                paths.plot(
                    *xy[:, case, core].T, color=color, label=name if core == 0 else None
                )
            paths.scatter(*xy[first, case].T, color=color, marker="x")
    if n == 193:
        impact_time = t[closest]
        paths.scatter(*xy[0, 0].T, marker="o", color="black", s=18, label="Start")
        paths.scatter([], [], marker="x", color="black", label="Separation threshold")
        gaps.scatter(
            t[[closest, farthest]],
            distance[[closest, farthest], 1],
            color="#b04729",
            zorder=4,
        )
paths.axhline(1, color="black")
paths.set(
    xlim=(0, 1),
    ylim=(0.48, 1.02),
    aspect="equal",
    xlabel="x / L",
    ylabel="y / L",
    title="Primary core trajectories",
)
paths.legend(fontsize=8)
gaps.set(
    xlabel="Time", ylabel="Core distance from wall / L", title="Approach and rebound"
)
gaps.legend(fontsize=8)
plt.show()

Both grids reach the divergence threshold at **$t=0.62$**. On the 193² grid the no-slip wall distance rises from $0.075L$ at $t=0.59$ to $0.129L$ in the final frame. Using unrounded positions, the observed rebound is **$0.055L$** (193²) and **$0.054L$** (129²).

## 7. Impact concentrates viscous dissipation near the wall

We measure viscous energy loss in the **top sixteenth of the box**, the strip $A$ extending from $y=15L/16$ to the upper wall at $y=L$. On both grids we sample the local dissipation rate $\epsilon$ every $0.01$ time units. Integrating over the strip gives its loss rate $D_A$. Integrating over time gives its accumulated loss $Q_A$.
$$
\epsilon=2\nu\sum_{i,j} S_{ij}^2,\qquad S_{ij}=\tfrac12(\partial_i u_j+\partial_j u_i).
$$
$$
D_A(t)=\int_A \epsilon(x,y,t)\,dA,\qquad Q_A(t)=\int_0^t D_A(s)\,ds.
$$
Here $S_{ij}$ measures local deformation and $\epsilon$ is the rate at which viscosity removes kinetic energy per unit mass. $D_A$ adds those losses over the strip at a single instant. $Q_A$ is the area under the $D_A$ curve up to time $t$. Fluid can also carry kinetic energy across its lower edge.

The dashed line in all three panels marks **no-slip closest approach** at $t=0.59$, the saved instant when the two no-slip cores on the fine grid have their smallest mean distance from the top wall (Section 6). The cores stay inside the fluid. We use this same time reference for both runs.

In [ ]:
from scipy.integrate import cumulative_trapezoid

dissipation_runs = {nodes: compare(nodes) for nodes in (129, 193)}
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.4), sharex=True)
fig.subplots_adjust(left=0.065, right=0.98, bottom=0.27, top=0.77, wspace=0.34)

for nodes, (times, rates, energies) in dissipation_runs.items():
    for index, (wall, color) in enumerate(
        (("Free slip", "#126987"), ("No slip", "#b04729"))
    ):
        total_rate, strip_rate = rates[:, index, :].T
        strip_loss = cumulative_trapezoid(strip_rate, x=times, initial=0)
        style = dict(color=color, linewidth=1.9, label=f"{wall}, {nodes}²")
        if nodes == 129:
            style.update(
                linestyle="none", marker="o", markerfacecolor="none", markevery=4
            )
        axes[0].plot(times, strip_rate, **style)
        axes[1].plot(times, 100 * strip_rate / total_rate, **style)
        axes[2].plot(times, 100 * strip_loss / energies[0, index], **style)

titles = (
    "Dissipation in the strip",
    "Share of total dissipation",
    "Energy dissipated in the strip",
)
units = (
    "Energy loss per unit time",
    "Instantaneous share (%)",
    "% of initial kinetic energy",
)
for ax, title, ylabel in zip(axes, titles, units):
    ax.axvline(
        impact_time,
        color="0.3",
        linestyle="--",
        linewidth=1,
        label=f"No-slip closest approach (t={impact_time:.2f})",
        zorder=1,
    )
    ax.set(
        title=title, ylabel=ylabel, xlabel="Time", xlim=(0, times[-1]), ylim=(0, None)
    )
    ax.grid(alpha=0.2)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle("Vortex–wall collision  |  Within L/16 of the upper wall")
plt.show()

**Left — loss rate in the strip.** At $t=0.55$, the no-slip movie already shows thin bands above the approaching cores. In the Section 5 profile through the blue band, leftward velocity rises sharply to zero at the wall. This shear contributes approximately $\nu(\partial_y u)^2$ to the local dissipation, so both bands contribute even though their vorticities have opposite signs.

On the fine grid, the strip loss rate peaks at $t=0.56$, about **2.2 times** the free-slip peak. The cores continue toward the wall until $t=0.59$. The timing makes sense because the loss rate responds to strain throughout the strip, while closest approach depends on the positions of the core centers. The profile shows that strong shear has developed during approach. The time of maximum strip loss comes from the integrated rate curve.

**Middle — share of whole-box dissipation.** $D_A/D_\Omega$ gives the fraction of the instantaneous loss occurring in the top sixteenth. The denominator $D_\Omega$ includes the whole box at the same time. A value above $6.25\%$, the strip’s share of box area, means dissipation per unit area is higher in the strip than in the box on average.

**Right — accumulated loss.** The plotted percentage is $100Q_A(t)/K(0)$, where $K(0)$ is the initial kinetic energy of the whole box. A steeper curve means faster loss in the strip. The free-slip rate overtakes the no-slip rate around $t=0.78$, but the earlier no-slip burst contributes more to the time integral. By $t=1.20$, the strip has dissipated **17.46%** of the box’s initial kinetic energy with no slip and **13.20%** with free slip on the fine grid.

### 7.1 Cumulative loss and energy balance on both grids

The last column checks the whole-box budget, $[K(t)+Q_\Omega(t)-K(0)]/K(0)$. The stationary walls supply zero power, so the continuum residual is zero.

In [ ]:
rows = [
    "| Nodes per side | Upper wall | Peak strip rate | Strip loss / initial energy | Strip loss / total loss | Signed budget residual / initial energy |",
    "|---:|:---|---:|---:|---:|---:|",
]
for nodes, (times, rates, energies) in dissipation_runs.items():
    for index, wall in enumerate(("Free slip", "No slip")):
        total_rate, strip_rate = rates[:, index, :].T
        total_loss = trapezoid(total_rate, x=times)
        strip_loss = trapezoid(strip_rate, x=times)
        initial = energies[0, index]
        residual = (energies[-1, index] + total_loss - initial) / initial
        rows.append(
            f"| {nodes} | {wall} | {strip_rate.max():.5f} | {strip_loss / initial:.2%} "
            f"| {strip_loss / total_loss:.2%} | {residual:+.2%} |"
        )
display(Markdown("\n".join(rows)))

The coarse grid gives the same ordering (17.30% with no slip, 13.18% with free slip). For this collision, the wall effect exceeds the change under refinement. The fine-grid no-slip budget residual has magnitude 0.51% of initial energy. [Setup](simulations/run_2d_wall_collision.py) · [Dissipation calculation](simulations/run_2d_wall_dissipation.py)

## 8. Solver validation

**Free-slip decay.** A single vorticity mode decays as $\exp(-\nu|\mathbf{k}|^2t)$, where $\mathbf{k}$ is its wavevector. Halving the grid spacing reduces the final field error by about four, as expected for second-order spatial accuracy. [Decay and grid-refinement plot](figures/walls/free_slip_decay.png)

**No-slip cavity.** A moving lid drives circulation at Reynolds number 100. On the finer grid the maximum discrepancy across the two steady centerline velocity profiles is 0.59% of lid speed. This checks that the solver transmits wall motion into the interior. [Cavity benchmark plot](figures/walls/cavity_profiles.png) · [Ghia, Ghia, and Shin, Tables I and II](#ref-ghia)

## 9. References

1. <a id="ref-kramer"></a>W. Kramer, H. J. H. Clercx, and G. J. F. van Heijst (2007). “Vorticity dynamics of a dipole colliding with a no-slip wall.” *Physics of Fluids* **19**(12), 126603. [DOI](https://doi.org/10.1063/1.2814345)

2. <a id="ref-ghia"></a>U. Ghia, K. N. Ghia, and C. T. Shin (1982). “High-Re solutions for incompressible flow using the Navier-Stokes equations and a multigrid method.” *Journal of Computational Physics* **48**(3), 387–411. [DOI](https://doi.org/10.1016/0021-9991%2882%2990058-4)